In [ ]:
!pip install torch torch-geometric
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader


# 1. Define input and output directories
input_folder = Path("results/step_05")
output_folder = Path("results/step_06")

output_folder.mkdir(
    parents=True,
    exist_ok=True
)


# 2. Load the forecasting windows
with np.load(
    input_folder / "forecasting_windows.npz",
    allow_pickle=False
) as arrays:

    X_train = arrays["X_train"]
    y_train = arrays["y_train"]
    y_train_raw = arrays["y_train_raw"]

    X_val = arrays["X_val"]
    y_val = arrays["y_val"]
    y_val_raw = arrays["y_val_raw"]

    X_test = arrays["X_test"]
    y_test = arrays["y_test"]
    y_test_raw = arrays["y_test_raw"]

    training_origins = arrays["training_origins"]
    validation_origins = arrays["validation_origins"]
    test_origins = arrays["test_origins"]

    references = arrays["references"].tolist()
    feature_names = arrays["feature_names"].tolist()

    target_scale = float(
        arrays["target_scale"].item()
    )

encoder_length = X_train.shape[1]
n_parts = X_train.shape[2]
n_features = X_train.shape[3]
forecast_horizon = y_train.shape[2]

n_nodes_per_graph = (
    encoder_length * n_parts
)


# 3. Load and align the part metadata
part_metadata = pd.read_csv(
    input_folder / "part_metadata.csv"
)

if part_metadata["Reference"].duplicated().any():
    raise ValueError(
        "Duplicate part references were found in the metadata."
    )

part_metadata = (
    part_metadata
    .set_index("Reference")
    .loc[references]
    .reset_index()
)

if part_metadata["Subsystem"].isna().any():
    raise ValueError(
        "Every part must have a subsystem."
    )

subsystems = (
    part_metadata["Subsystem"]
    .to_numpy()
)


# 4. Define a consistent node numbering scheme
# Node ID = historical month position × number of parts + part position.
def node_id(month_position, part_position):

    return (
        month_position * n_parts
        + part_position
    )


# 5. Construct temporal and relational edges
temporal_edges = []
relational_edges = []

for month_position in range(encoder_length):

    # Connect different parts from the same subsystem
    # within the same historical month.
    for source_part in range(n_parts):

        for target_part in range(n_parts):

            if (
                source_part != target_part
                and subsystems[source_part]
                == subsystems[target_part]
            ):

                relational_edges.append((
                    node_id(
                        month_position,
                        source_part
                    ),
                    node_id(
                        month_position,
                        target_part
                    )
                ))

    # Connect the same part across consecutive historical months.
    if month_position > 0:

        for part_position in range(n_parts):

            previous_node = node_id(
                month_position - 1,
                part_position
            )

            current_node = node_id(
                month_position,
                part_position
            )

            # Both endpoints belong to the observed encoder history.
            temporal_edges.append((
                previous_node,
                current_node
            ))

            temporal_edges.append((
                current_node,
                previous_node
            ))

all_edges = (
    temporal_edges + relational_edges
)

edge_index = (
    torch.tensor(
        all_edges,
        dtype=torch.long
    )
    .t()
    .contiguous()
)

# Self-loops are not explicitly added here.
# GCNConv and GATConv add them internally by default.


# 6. Identify the nodes at the forecast origin
# These are the nodes representing the last historical month.
last_month_mask = torch.zeros(
    n_nodes_per_graph,
    dtype=torch.bool
)

last_month_mask[
    (encoder_length - 1) * n_parts:
] = True


# 7. Check the graph topology
assert edge_index.shape[0] == 2
assert edge_index.min().item() >= 0

assert (
    edge_index.max().item()
    < n_nodes_per_graph
)

assert (
    last_month_mask.sum().item()
    == n_parts
)

for source, target in temporal_edges:

    source_month, source_part = divmod(
        source,
        n_parts
    )

    target_month, target_part = divmod(
        target,
        n_parts
    )

    assert source_part == target_part
    assert abs(source_month - target_month) == 1

for source, target in relational_edges:

    source_month, source_part = divmod(
        source,
        n_parts
    )

    target_month, target_part = divmod(
        target,
        n_parts
    )

    assert source_month == target_month
    assert source_part != target_part

    assert (
        subsystems[source_part]
        == subsystems[target_part]
    )


# 8. Define the graph dataset construction function
def build_graph_dataset(
    input_windows,
    target_windows,
    raw_target_windows,
    origins
):

    graphs = []

    for window_index, origin in enumerate(origins):

        # Flatten the historical month and part dimensions.
        # Shape: [historical nodes, input features].
        node_features = input_windows[
            window_index
        ].reshape(
            n_nodes_per_graph,
            n_features
        )

        # Labels remain separate from node input features.
        # Shape: [parts, six future horizons].
        targets = target_windows[
            window_index
        ]

        raw_targets = raw_target_windows[
            window_index
        ]

        graph = Data(
            x=torch.tensor(
                node_features,
                dtype=torch.float32
            ),
            edge_index=edge_index,
            y=torch.tensor(
                targets,
                dtype=torch.float32
            ),
            y_raw=torch.tensor(
                raw_targets,
                dtype=torch.float32
            ),
            last=last_month_mask.clone(),
            origin=torch.tensor(
                [int(origin)],
                dtype=torch.long
            )
        )

        assert graph.x.shape == (
            n_nodes_per_graph,
            n_features
        )

        assert graph.y.shape == (
            n_parts,
            forecast_horizon
        )

        assert torch.isfinite(graph.x).all()
        assert torch.isfinite(graph.y).all()

        graphs.append(graph)

    return graphs


# 9. Create the training, validation, and test graph datasets
train_graphs = build_graph_dataset(
    X_train,
    y_train,
    y_train_raw,
    training_origins
)

validation_graphs = build_graph_dataset(
    X_val,
    y_val,
    y_val_raw,
    validation_origins
)

test_graphs = build_graph_dataset(
    X_test,
    y_test,
    y_test_raw,
    test_origins
)


# 10. Verify graph batching
# PyTorch Geometric automatically offsets node IDs
# when several graphs are combined into one batch.
check_loader = DataLoader(
    train_graphs[:2],
    batch_size=2,
    shuffle=False
)

check_batch = next(
    iter(check_loader)
)

assert check_batch.num_graphs == 2

assert check_batch.x.shape == (
    2 * n_nodes_per_graph,
    n_features
)

assert check_batch.y.shape == (
    2 * n_parts,
    forecast_horizon
)

assert (
    check_batch.last.sum().item()
    == 2 * n_parts
)


# 11. Save a readable edge table
edge_records = []

for edge_type, edges in [
    ("Temporal", temporal_edges),
    ("Relational", relational_edges)
]:

    for source, target in edges:

        edge_records.append({
            "Source_Node": source,
            "Target_Node": target,
            "Edge_Type": edge_type
        })

edge_table = pd.DataFrame(
    edge_records
)

edge_table.to_csv(
    output_folder / "graph_edges_template.csv",
    index=False
)


# 12. Save a readable node template
node_records = []

for month_position in range(encoder_length):

    for part_position, reference in enumerate(references):

        node_records.append({
            "Node_ID": node_id(
                month_position,
                part_position
            ),
            "Historical_Month_Position": month_position,
            "Reference": reference,
            "Subsystem": subsystems[part_position],
            "Is_Last_Historical_Month": (
                month_position == encoder_length - 1
            )
        })

node_table = pd.DataFrame(
    node_records
)

node_table.to_csv(
    output_folder / "graph_nodes_template.csv",
    index=False
)


# 13. Save tensors and metadata
# Graphs are serialized as dictionaries of tensors.
# They can later be reconstructed using Data(**graph_dictionary).
graph_payload = {
    "train": [
        graph.to_dict()
        for graph in train_graphs
    ],
    "validation": [
        graph.to_dict()
        for graph in validation_graphs
    ],
    "test": [
        graph.to_dict()
        for graph in test_graphs
    ],
    "references": references,
    "feature_names": feature_names,
    "target_scale": target_scale,
    "encoder_length": encoder_length,
    "forecast_horizon": forecast_horizon,
    "n_parts": n_parts,
    "n_features": n_features
}

torch.save(
    graph_payload,
    output_folder / "graph_datasets.pt"
)


# 14. Save a graph construction summary
summary = {
    "training_graphs": len(train_graphs),
    "validation_graphs": len(validation_graphs),
    "test_graphs": len(test_graphs),
    "nodes_per_graph": n_nodes_per_graph,
    "features_per_node": n_features,
    "temporal_directed_edges": len(temporal_edges),
    "relational_directed_edges": len(relational_edges),
    "total_directed_edges": len(all_edges),
    "prediction_nodes_per_graph": n_parts,
    "forecast_horizon": forecast_horizon,
    "self_loops": (
        "Added internally by GCNConv or GATConv"
    ),
    "causality": (
        "All input nodes belong to the observed historical window. "
        "Future quantities are stored as labels only."
    )
}

(
    output_folder / "graph_construction_summary.json"
).write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)


# 15. Display the results
print("Graph datasets created successfully.")

print("\nNumber of training graphs:", len(train_graphs))
print("Number of validation graphs:", len(validation_graphs))
print("Number of test graphs:", len(test_graphs))

print("\nNodes per graph:", n_nodes_per_graph)
print("Features per node:", n_features)
print("Temporal directed edges:", len(temporal_edges))
print("Relational directed edges:", len(relational_edges))

print("\nFirst training graph:")
print(train_graphs[0])

print("\nBatched input shape:", tuple(check_batch.x.shape))
print("Batched target shape:", tuple(check_batch.y.shape))

print(f"\nFiles saved in: {output_folder}")
